In [1]:
import os
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["OPENWEATHER_API_KEY"] = os.getenv("OPENWEATHER_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [2]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
output_parser = StrOutputParser()

# response = llm.invoke("Hey! Tell me something about you")
# print(response.content)

In [3]:
# Tool 1: Get City Latitude and Longitude
@tool
def get_city_lat_lon(city: str, country_code: str) -> tuple:
    """
    Retrieves the latitude and longitude of a specified city using the OpenWeather API.
    """
    api_key = os.environ["OPENWEATHER_API_KEY"]
    url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country_code}&limit=1&appid={api_key}&mode=JSON"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data[0]["lat"], data[0]["lon"]
    else:
        return None, None

In [4]:
# lat, lon = get_city_lat_lon(country_code="BD", city="dhaka")

# print(f"lat: {lat}\nlon: {lon}")

In [5]:
# Tool 2: Get Weather Information
@tool
def get_weather(lat: float, lon: float) -> str:
    """
    Retrieves the current weather information for the given latitude and longitude using the OpenWeather API.
    """
    api_key = os.environ["OPENWEATHER_API_KEY"]
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        description = data['weather'][0]['description']
        temperature = data['main']['temp']
        humidity = data['main']['humidity']
        wind_speed = data['wind']['speed']
        return (f"Weather: {description}\n"
                f"Temperature: {temperature} ℃\n"
                f"Humidity: {humidity}%\n"
                f"Wind Speed: {wind_speed} m/s")
    else:
        return "Unable to retrieve weather data."

In [6]:
# get_weather(lat, lon)

In [7]:
# # Defining the prompt
# prompt = ChatPromptTemplate.from_template("""
# You are a smart weather assistant. When given a city name, intelligently identify the corresponding country code without asking the user. 
# If you're unsure, use common knowledge to make the best guess.

# Question: {question}
# """)

# question = "What is the current weather of dhaka"

In [8]:
# # Binding tools
# tools = [get_city_lat_lon, get_weather]
# llm_with_tools = llm.bind_tools(tools)

# chain = prompt | llm_with_tools

# response = chain.invoke(question)

# print(response)

In [9]:

# # # Creating agent and executor
# # agent = create_tool_calling_agent(llm_with_tools, tools)
# # agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# # Execute the agent
# response = chain.invoke(question)

# print(response)

In [13]:
prompt = ChatPromptTemplate.from_template("""
You are a smart weather assistant. Use the tools to provide accurate weather information.  Insted of giving the only weather information add some creative descussion.
Suggest the use if this weather is good for travel or not.

If you receive only the city name, determine the country code by yourself without asking the user. If unsure, use common knowledge to make the best guess.

Question: {question}

{agent_scratchpad}
""")


In [14]:
question = "What is the current weather of feni"

In [15]:
# Binding tools
tools = [get_city_lat_lon, get_weather]
llm_with_tools = llm.bind_tools(tools)

# Creating the ReAct agent
agent = create_tool_calling_agent(llm_with_tools, tools, prompt)

# Agent Executor to manage tool execution
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Now invoking the agent
response = agent_executor.invoke({"question": question})

print(response['output'])  # To get the final response content




> Entering new AgentExecutor chain...

Invoking: `get_city_lat_lon` with `{'city': 'feni', 'country_code': 'BD'}`


(23.0068161, 91.3965276)
Invoking: `get_weather` with `{'lon': 91.3965276, 'lat': 23.0068161}`


Weather: clear sky
Temperature: 27.16 ℃
Humidity: 23%
Currently in Feni, Bangladesh, the weather is clear with a temperature of 27.16°C. The humidity is at 23%, and the wind speed is 2.78 m/s.

With clear skies and a moderate temperature, it sounds like a pleasant day for traveling. The low humidity should make it quite comfortable. Whether it's a local trip or something more extensive, the weather seems cooperative!

> Finished chain.
Currently in Feni, Bangladesh, the weather is clear with a temperature of 27.16°C. The humidity is at 23%, and the wind speed is 2.78 m/s.

With clear skies and a moderate temperature, it sounds like a pleasant day for traveling. The low humidity should make it quite comfortable. Whether it's a local trip or something more extensive, the weath